# Behaviour cloning — Notebook 2

You built an autograd engine by hand. Now you use a real one, and train a
network to imitate the scripted controller.

**How this one works.** Almost no code is given to you. Each section is a
short reading, then decisions you make yourself, then a CHECK that verifies
whatever you built actually does what you think it does. Where I have an
opinion I'll say so, but several of these have no single right answer and
the point is that you pick and find out.

Run this locally in the venv (`jupyter notebook` from `C:\game`), not Colab —
it imports `rocketenv` and loads the dataset you generate.

**Prerequisite:** a dataset from the scripted expert. The notebook assumes
you can load it into two arrays — observations `(N, 13)` and actions
`(N, 2)`, both float32 — plus whatever else you decided to record.

---
## 1. Tensors — same autograd, coarser granularity

Your `Value` held one number and one node in the graph per scalar operation.
A tensor holds a whole array, and each node covers an entire array operation.
The graph has the *same shape* — it's the granularity that changed.

That difference is the entire reason this is fast. Your `MLP(3,[4,4,1])`
built roughly 200 Python objects per forward pass. The equivalent
`nn.Linear` does one matrix multiply: a single dispatch into optimised
C/BLAS, with no per-scalar Python overhead. Same math, ~1000× the throughput.

Everything you implemented is present under names you'll recognise:

| yours | PyTorch |
|---|---|
| `Value(x)` | `torch.tensor(x, requires_grad=True)` |
| `.grad` | `.grad` |
| `.backward()` | `.backward()` |
| `zero the grads` | `optimizer.zero_grad()` |
| `p.data -= lr * p.grad` | `optimizer.step()` |

`requires_grad=True` is the flag that says "this is a leaf I want gradients
for" — the parameters. Everything computed from it joins the graph
automatically, exactly like your `_prev` links.

**Exercise.** Reproduce the Section 4 graph from Notebook 1 in PyTorch:
`x = 3.0`, `y = -2.0`, `L = (x*y + x) * 2.0`. Get `x.grad` and `y.grad`.
You hand-computed these; confirm PyTorch agrees.

In [ ]:
import torch

# YOUR CODE HERE: build x, y as tensors with requires_grad=True,
# compute L, call backward, and read the gradients.

In [ ]:
# --- CHECK 1 ---
assert x.grad is not None, "no gradient - did you set requires_grad=True?"
assert abs(x.grad.item() - (-2.0)) < 1e-6, f"x.grad should be -2.0, got {x.grad.item()}"
assert abs(y.grad.item() - 6.0) < 1e-6, f"y.grad should be 6.0, got {y.grad.item()}"
print("PASS - same numbers you derived by hand in Notebook 1")

---
## 2. `nn.Module` — what it does that your class didn't

Your `MLP` needed an explicit `parameters()` that walked layers and neurons
collecting Values. `nn.Module` does that automatically: any `nn.Parameter`
or sub-module you assign to `self` gets registered, and `.parameters()`
comes for free.

It also gives you things you didn't build:

- **`.train()` / `.eval()`** — switches behaviour of layers like dropout and
  batchnorm. You have neither, so it changes nothing for you *yet*; get in
  the habit anyway, because forgetting `.eval()` at inference is a classic
  silent bug.
- **`.to(device)`** — moves every parameter at once.
- **`state_dict()` / `load_state_dict()`** — save and load weights. You need
  this to get a model into AUTO mode.

`nn.Linear(in, out)` is your whole `Layer`: a weight matrix, a bias, and the
matmul. Note it does *not* include the activation — in PyTorch the
nonlinearity is a separate step, unlike your `Neuron` which had `tanh` baked
in.

**Exercise.** Write your policy network as an `nn.Module`: 13 inputs → some
hidden layers → 2 outputs. Two hidden layers of 64–128 is a reasonable
starting point, and it's the size the eventual PPO policy will use.

Leave the output raw for now — the next section is about what to do with it.

In [ ]:
import torch.nn as nn

# YOUR CODE HERE: class Policy(nn.Module) with __init__ and forward

In [ ]:
# --- CHECK 2 ---
_net = Policy()
_x = torch.randn(7, 13)          # a batch of 7 observations
_y = _net(_x)
assert _y.shape == (7, 2), f"batch of 7 obs should give shape (7, 2), got {tuple(_y.shape)}"
_n = sum(p.numel() for p in _net.parameters())
assert _n > 100, f"only {_n} parameters - is the network wired up?"
assert _net(torch.randn(1, 13)).shape == (1, 2), "should handle a batch of 1"
print(f"PASS - {_n} parameters, and it batches")

---
## 3. The action head — your first real design decision

`throttle` lives in `[0, 1]`. `gimbal` lives in `[-1, 1]`. Your network
currently emits two unbounded numbers. You have to decide what happens
between those two facts.

Three families, each with a real failure mode:

**Bounded activations** — `sigmoid` on the first output, `tanh` on the
second. Invalid actions become unrepresentable. But look at your dataset
first: how often does the expert command exactly `1.0` throttle? Saturating
activations approach their limits asymptotically and their gradient goes to
zero as they do, so targets *at* the boundary are both unreachable and
slow to learn toward.

**Unbounded output, clip at execution.** Gradients never saturate and
boundary targets are easy to hit. But the network can represent actions that
don't exist, and the loss will happily reward it for predicting `1.4`
throttle when the target was `1.0`, because it never sees the clipping.

**Unbounded, and just let MSE handle it.** Simplest. The training data only
ever contains valid actions, so the network has no reason to leave the range
by much — until it hits an unfamiliar state.

There's also a scaling question hiding here: the two outputs have different
ranges, so identical numerical error in each is not equally consequential.

**Decide, and write down why.** Then check what the expert's action
distribution actually looks like before you commit — that's the evidence
that settles the saturation question.

In [ ]:
# YOUR CODE HERE: look at your dataset's action distribution before deciding.
# Two histograms, or just min/max/mean/std and the fraction at the boundaries.

In [ ]:
# --- CHECK 3 ---
# Whatever you chose, verify it holds under inputs far outside the training
# distribution - that is exactly where a policy ends up when it drifts.
_wild = torch.randn(500, 13) * 8.0
with torch.no_grad():
    _a = _net(_wild)
print("throttle range:", _a[:, 0].min().item(), "to", _a[:, 0].max().item())
print("gimbal   range:", _a[:, 1].min().item(), "to", _a[:, 1].max().item())
print()
print("If those exceed the action space, that is a choice - make sure it was")
print("yours, and that something downstream clips before the env sees it.")

---
## 4. Loss and data — decisions, not defaults

**The loss.** MSE on the two action components is the obvious choice, and
probably correct. But `nn.MSELoss()` averages both dimensions equally, and
that's an assumption: it says one unit of throttle error is worth exactly
one unit of gimbal error. Is that true for landing a rocket? You know the
physics better than the loss function does.

Worth checking either way: what MSE does a network that ignores its input
and always predicts the dataset mean achieve? That number is your floor —
anything near it means you've learned nothing, regardless of how small it
looks in absolute terms.

**The split.** You need held-out data to distinguish learning from
memorising. I've already told you what I think the trap is with trajectory
data, so rather than take my word: **do it both ways.** Split randomly over
all samples, and split by whole episodes. Train the same network on each.
Compare the validation losses.

If they come out roughly equal, I was wrong and you should tell me. If one
is dramatically lower, work out which one is lying to you and why.

**Batching.** You did full-batch gradient descent in Notebook 1 — all four
examples, one update. With 100k samples that's both slow and worse:
minibatches give you many noisy updates per pass instead of one exact one,
and the noise helps escape bad regions. 64–256 is the usual range.

`torch.utils.data.TensorDataset` and `DataLoader` handle the batching and
shuffling. Or index into the arrays yourself with a permutation — it's about
five lines and you'll understand it better.

In [ ]:
# YOUR CODE HERE: load the dataset, compute the mean-predictor baseline loss,
# and build your train/validation splits (both ways).

In [ ]:
# --- CHECK 4 ---
# Adapt the names to whatever you used. This verifies the episode-level split
# actually separated episodes, which is the part that is easy to get subtly wrong.
_tr, _va = set(train_episode_ids), set(val_episode_ids)
assert _tr and _va, "both splits need episodes in them"
assert not (_tr & _va), f"{len(_tr & _va)} episodes appear in BOTH splits"
_frac = len(_va) / (len(_tr) + len(_va))
assert 0.05 < _frac < 0.4, f"validation is {_frac:.0%} of episodes - unusual, intentional?"
print(f"PASS - {len(_tr)} train / {len(_va)} val episodes, no overlap")

---
## 5. The training loop

Same four steps you wrote by hand, plus an outer loop over batches:

1. forward the batch, compute loss
2. `optimizer.zero_grad()`
3. `loss.backward()`
4. `optimizer.step()`

Steps 2–4 are literally your Notebook 1 loop with the manual parameter walk
replaced by an object that holds a reference to `model.parameters()`.

**Adam vs SGD.** You implemented SGD: `p -= lr * grad`. Adam keeps a running
estimate of each parameter's gradient mean and variance and scales the step
per-parameter, so parameters with consistently small gradients still move.
In practice it converges faster with less learning-rate tuning, which is why
it's the default for almost everything. `lr=1e-3` is the standard starting
point. Use `torch.optim.Adam`.

**Track both losses per epoch.** Train loss falling while validation loss
rises is overfitting, and you want to see it happen rather than read about
it. Plot them if you like — matplotlib is installed with jupyter.

**Then save the weights.** `torch.save(model.state_dict(), path)`. AUTO mode
will need it.

In [ ]:
# YOUR CODE HERE: the training loop. Track train and val loss per epoch.

In [ ]:
# --- CHECK 5 ---
assert len(train_losses) == len(val_losses) > 1, "track both, per epoch"
assert val_losses[-1] < val_losses[0], "validation loss did not improve at all"
_ratio = val_losses[-1] / mean_predictor_loss
assert _ratio < 0.5, (
    f"final val loss is {_ratio:.0%} of the mean-predictor baseline - "
    "the network is barely beating a constant")
print(f"PASS - val loss {val_losses[-1]:.5f}, "
      f"{_ratio:.1%} of the mean-predictor baseline")

---
## 6. The loss is not the metric

This is the section that matters.

Your validation MSE measures one thing: agreement with the expert **on
states the expert visited**. It does not measure whether the rocket lands.
Those come apart, and the mechanism is worth understanding because it's the
central weakness of behaviour cloning.

Suppose your policy is 99% accurate per step. Over an 800-step episode it
deviates slightly, which puts the rocket in a state marginally different
from any the expert produced. Your network has never been trained there, so
its error grows. That puts it somewhere stranger still. Errors compound
along the trajectory, and the states where you're worst are precisely the
ones your dataset doesn't cover.

Consequence: **two networks with identical validation loss can have wildly
different landing rates**, and the only way to know is to run the thing.

The harness below is provided — it's measurement plumbing, not a learning
exercise. It takes a function mapping one observation to one action.

In [ ]:
import numpy as np
from rocketenv import RocketEnv
from rocketenv.reward import TOUCHDOWN


def evaluate(policy_fn, n_episodes=50, terrain=None, options=None, seed0=10_000):
    '''Run a policy and report outcomes. policy_fn: (13,) float32 -> (2,) float32.

    Seeds start at 10_000 to keep evaluation episodes disjoint from any
    low-numbered seeds you used to generate training data.
    '''
    env = RocketEnv(terrain=terrain) if terrain is not None else RocketEnv()
    outcomes, fuels, impacts = [], [], []
    for i in range(n_episodes):
        obs, _ = env.reset(seed=seed0 + i, options=options)
        for _ in range(env.cfg.max_steps):
            obs, _, term, trunc, info = env.step(policy_fn(obs))
            if term or trunc:
                break
        s = info["state"]
        outcomes.append(info.get("outcome", "TIMEOUT"))
        fuels.append(s[6])
        impacts.append(float(np.hypot(s[2], s[3])))
    landed = sum(o == TOUCHDOWN for o in outcomes)
    from collections import Counter
    print(f"landed {landed}/{n_episodes} ({landed/n_episodes:.0%})")
    print("outcomes:", dict(Counter(outcomes)))
    print(f"mean impact {np.mean(impacts):.2f} m/s, mean fuel left {np.mean(fuels):.1f}")
    return landed / n_episodes

In [ ]:
# YOUR CODE HERE: wrap your trained model in a policy_fn and evaluate it.
# Remember .eval() and torch.no_grad(). The expert lands 50/50 on flat.

---
## 7. When it underperforms

It probably will, at least the first time — the gap between validation loss
and landing rate is the normal result here, not a bug.

Candidate explanations, and how you'd tell them apart:

- **Compounding error / distribution shift.** Diagnostic: log the states
  where it fails and compare them against your dataset. Are they states the
  expert never visited? Fix: more action noise during collection, or more
  data, or DAgger — collecting fresh data from the states *your policy*
  actually reaches, with the expert labelling them.
- **The observation is insufficient.** If two states share an observation
  but need different actions, no network can fit both. You already know
  where this bites on generated terrain.
- **Underfitting.** Compare train loss to the mean-predictor baseline. If
  it's not much better, the network or the training is the problem, not the
  data.
- **Action-head saturation.** If failures cluster at full throttle, revisit
  Section 3.

**Worth doing before you move on:** measure the landing rate against
dataset size. Train on 10 episodes, 30, 100, and plot. If it's still
climbing steeply at 100, you're data-limited and collecting more is the
cheapest fix. If it's flat, more data won't help and the problem is
elsewhere. That curve tells you which of the above you're looking at faster
than any amount of staring at loss values.

---

### What this bought you

PyTorch, a working supervised pipeline, and a policy you can load in AUTO
mode. Also the concrete experience of a proxy loss diverging from the metric
you care about.

And the honest limitation: BC can only reproduce the expert, never beat it,
and it degrades exactly where the expert never went. Fixing that requires
learning from *outcomes* rather than from labels — which is the entire point
of what comes next.